# درس ۰۱ - معرفی عامل‌های هوش مصنوعی

به اولین درس دوره **عامل‌های هوش مصنوعی برای مبتدیان** خوش آمدید!

یک **عامل هوش مصنوعی** برنامه‌ای است که از یک مدل زبان بزرگ (LLM) به‌عنوان موتور استدلال خود استفاده می‌کند و می‌تواند در دنیای واقعی *اقداماتی* انجام دهد — فراخوانی APIها، پرس‌وجو در پایگاه داده‌ها، یا اجرای کد — تا هدفی را به نمایندگی از کاربر محقق کند.

در این دفترچه یادداشت شما اولین عامل خود را می‌سازید: یک **عامل مسافرتی** که مقاصد تعطیلات را پیشنهاد می‌دهد. در این مسیر یاد خواهید گرفت چگونه:

1. به سرویس عامل Azure AI Foundry با استفاده از **چارچوب عامل مایکروسافت** متصل شوید.
2. به عامل یک **ابزار** بدهید — یک تابع ساده پایتون که می‌تواند آن را فراخوانی کند.
3. عامل را اجرا کرده و پاسخ آن را بررسی کنید.
4. پاسخ عامل را به صورت توکن به توکن پخش کنید.


## تنظیم

قبل از اجرای این دفترچه یادداشت، مطمئن شوید که:

1. **یک پروژه Azure AI Foundry** با یک مدل چت مستقر شده دارید (مثلاً `gpt-4o-mini`).
2. **با Azure CLI وارد شده‌اید** — دستور `az login` را در ترمینال خود اجرا کنید.
3. **متغیرهای محیطی مورد نیاز را تنظیم کرده‌اید:**
   - `AZURE_AI_PROJECT_ENDPOINT` — نقطه پایانی پروژه Azure AI Foundry شما.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — نام مدل مستقر شده شما.

سلول زیر بسته‌های پایتون مورد نیاز را نصب می‌کند.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## ایجاد اولین ایجنت خود

یک ایجنت به دو چیز نیاز دارد:

- **دستورالعمل‌ها** که به آن می‌گویند *که چه کسی است* و *چگونه رفتار کند* (یک پرامپت سیستمی).
- **ابزارها** — توابع پایتون که با `@tool` تزئین شده‌اند و ایجنت می‌تواند برای دریافت اطلاعات یا انجام اقدامات از آنها استفاده کند.

در ادامه، یک ابزار ساده تعریف می‌کنیم که فهرستی از مقاصد محبوب تعطیلات را بازمی‌گرداند. ایجنت از این ابزار زمانی استفاده خواهد کرد که کاربر درخواست پیشنهاد سفر کند.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## پاسخ‌های جریان‌یافته

برای تجربه‌ای تعاملی‌تر می‌توانید پاسخ عامل را به صورت **جریان‌یافته** دریافت کنید. به جای اینکه منتظر پاسخ کامل بمانید، عامل قطعات متن را به محض تولید کردن ارائه می‌دهد. این به‌ویژه در رابط‌های گفت‌وگو که می‌خواهید خروجی را در زمان واقعی نمایش دهید مفید است.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## خلاصه

در این درس یاد گرفتید چگونه:

- **یک فراهم‌کننده بسازید** که از طریق `AzureAIProjectAgentProvider` به سرویس Azure AI Foundry Agent متصل می‌شود.
- **یک ابزار تعریف کنید** با استفاده از دکوراتور `@tool` تا عامل بتواند توابع پایتون شما را فراخوانی کند.
- **عامل را اجرا کنید** با یک پیام کاربر و پاسخ آن را چاپ کنید.
- **پاسخ‌ها را به صورت جریان** برای خروجی بلادرنگ پخش کنید.

در درس بعدی، چارچوب‌های عامل‌محور را عمیق‌تر بررسی خواهیم کرد و یاد می‌گیریم چگونه ابزارهای قدرتمندتر و قابلیت‌های استدلال چندمرحله‌ای به عوامل بدهیم.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
